## Tomato Leaf Disease Detection using CNN

### 1. Setup and Data Download

First, we import the necessary libraries, including `kagglehub` to download the dataset, `tensorflow` for building the model, and `matplotlib` for plotting results. We then download the 'tomato-leaf-dataset-for-disease-detection' from Kaggle Hub.

In [ ]:
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os

# Download the dataset from Kaggle Hub
path = kagglehub.dataset_download("yusufmurtaza01/tomato-leaf-dataset-for-disease-detection")

print(f"Path to dataset files: {path}")

### 2. Data Loading and Configuration

We define configuration parameters such as `BATCH_SIZE` and `IMG_SIZE`. The dataset is loaded using `tf.keras.utils.image_dataset_from_directory`, splitting it into training and validation sets. We also adjust the path to point to the `tomato_yolo_dataset` subfolder if it exists.

In [ ]:
# Configuration
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

# The KaggleHub download might place the actual data in a subfolder. Check for it.
adjusted_path = os.path.join(path, 'tomato_yolo_dataset')
if not os.path.exists(adjusted_path):
    adjusted_path = path

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    adjusted_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    adjusted_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

### 3. Data Preprocessing and Optimization

We extract the class names from the training dataset. To improve training performance, the datasets are cached and prefetched using `tf.data.AUTOTUNE`.

In [ ]:
class_names = train_ds.class_names
print(f"Detected classes: {class_names}")

# Optimize for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

### 4. Build Convolutional Neural Network (CNN) Model

A sequential CNN model is constructed with convolutional layers (`Conv2D`), max pooling layers (`MaxPooling2D`), and dense layers (`Dense`). A `Rescaling` layer normalizes pixel values, and `Dropout` is included for regularization. The final dense layer uses 'sigmoid' activation for binary classification (if `num_classes == 1`) or 'softmax' for multi-class classification.

In [ ]:
# Build CNN Model
num_classes = len(class_names)
# If num_classes is 1, we use sigmoid; otherwise softmax.
activation_fn = 'sigmoid' if num_classes == 1 else 'softmax'

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation=activation_fn)
])

model.summary()

### 5. Compile the Model

The model is compiled using the `adam` optimizer and an appropriate loss function (`binary_crossentropy` for binary classification or `sparse_categorical_crossentropy` for multi-class). `accuracy` is used as the evaluation metric.

In [ ]:
# Compile the model
loss_fn = 'binary_crossentropy' if num_classes == 1 else 'sparse_categorical_crossentropy'

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)

### 6. Train the Model

The model is trained on the `train_ds` dataset and validated on `val_ds` for a specified number of `EPOCHS` (10 in this case). The training history is stored.

In [ ]:
# Train the model
EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

### 7. Plot Training Results

Finally, the training and validation accuracy and loss values over epochs are plotted to visualize the model's performance during training.

In [ ]:
# Plot results
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()